In [1]:
import os

In [ ]:
%pwd

In [2]:
os.chdir('../')

In [ ]:
%pwd

In [3]:
from pydantic import BaseModel, ConfigDict
from pathlib import Path

class DataIngestionConfig(BaseModel):
    model_config = ConfigDict(extra='forbid')

    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path


class Config(BaseModel):
    model_config = ConfigDict(extra='forbid')

    artifacts_root: Path
    data_ingestion: DataIngestionConfig

In [4]:
from src.end_to_end_deep_learning_project_yt.constants import constants
from src.end_to_end_deep_learning_project_yt.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(
        self
        , config_filepath = constants.CONFIG_FILE_PATH
        , params_filepath = constants.PARAMS_FILE_PATH
    ) -> None:
        self.config = read_yaml(config_filepath, Config)
        # self.params = read_yaml(params_filepath, DataIngestionConfig)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        return config

In [6]:
import os
import urllib.request as request
import zipfile
from src.end_to_end_deep_learning_project_yt.utils.custom_logger import logger
from src.end_to_end_deep_learning_project_yt.utils.common import get_size, create_directories

In [7]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig) -> None:
        self.config = config


    def download_file(self) -> None:
        target = Path(self.config.local_data_file)
        if target.exists():
            logger.info(f'File already exists of size: {get_size(Path(self.config.local_data_file))}')
        else:
            create_directories([target.parent])
            logger.info(f'created directory at: {target.parent}')

            filename, headers = request.urlretrieve(
                url = self.config.source_url
                , filename = target
            )
            logger.info(f'{filename} downloaded with following info: \n{headers}')


    def extract_zip_file(self) -> None:
        '''
        Extracts the zip file into data directory

        args:
            zip_file_path: str

        returns:
            None
        '''
        unzip_path = Path(self.config.unzip_dir)

        create_directories([unzip_path])
        logger.info(f'created directory at: {unzip_path}')

        with zipfile.ZipFile(self.config.local_data_file) as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f'zip extracted at: {unzip_path}')

In [8]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-10-31 12:14:42,940]: INFO - common - yaml file: config/config.yaml loaded successfully
[2026-10-31 12:14:42,942]: INFO - 2089676421 - created directory at: artifacts/data_ingestion
[2026-10-31 12:14:47,542]: INFO - 2089676421 - artifacts/data_ingestion/data.zip downloaded with following info: 
Connection: close
Content-Length: 11616915
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "adf745abc03891fe493c3be264ec012691fe3fa21d861f35a27edbe6d86a76b1"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 7396:1D114C:42D2AB:5677E7:6AE5D92D
x-github-edge-region: iad
Accept-Ranges: bytes
Date: Sat, 31 Oct 2026 12:14:45 GMT
Via: 1.1 varnish
X-Served-By: cache-iad-khef600067-IAD
X-Cache: HIT
X-Cache-Hits: 0
X-Timer: S1793448885.999573,VS0,VE1
Vary: Authorization,Accept-Encoding
Access-Control-Allow-